# Import libraries

In [ ]:
import json
import numpy as np
import pandas as pd
from helpers.phenotype import attach_populations
from sklearn.decomposition import PCA
from helpers.vizualisation import build_deme_colors, visualize_grid, visualize_pca, visualize_multiple_plots
from helpers.grid import create_grid
from scipy.cluster.hierarchy import linkage, fcluster, dendrogram
from scipy.spatial.distance import pdist
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler
from sklearn.decomposition import PCA

with open("params.json") as f:
    params = json.load(f)
k           = params["k"]
Ne          = params["Ne"]
M           = params["M"]
n_per_deme  = params["n_per_deme"]
n_loci      = params["n_loci"]
n_mutations = params["n_mutations"]
random_seed = params["random_seed"]
F           = params.get("F", 0.1)

fname = f"data/G_k{k}_Ne{Ne}_M{M}_npd{n_per_deme}_nloc{n_loci}_seed{random_seed}.npz"
data  = np.load(fname)
G     = data["G"]

humans = attach_populations(G.shape[1], k, n_per_deme)

G_T = G.T

results = {}
pops = humans['populations'].values  # length 2000

for pop in np.unique(pops):
    mask = pops == pop
    G_pop = G_T[mask]           # subset individuals for this population
    # Pick one random individual per SNP and flip
    idxs = np.random.randint(0, G_pop.shape[0], size=G_pop.shape[1])
    G_pop[idxs, np.arange(G_pop.shape[1])] = 1 - G_pop[idxs, np.arange(G_pop.shape[1])]
    freq = G_pop.mean(axis=0)   # frequency of 1s per SNP
    results[pop] = freq

freq_df = pd.DataFrame(results).T

def generate_genotypes(freq_df, F, n_individuals):
    dfs = []
    for pop, row in freq_df.iterrows():
        q = row.values                       # minor allele freq
        p = 1 - q
        prob_major = p**2 + F * p * q
        prob_het   = 2 * p * q * (1 - F)
        r = np.random.uniform(size=(n_individuals, len(q)))
        genotypes = np.where(r < prob_major, 1,
                    np.where(r < prob_major + prob_het, 0, -1))
        pop_df = pd.DataFrame(genotypes, columns=freq_df.columns)
        pop_df['population'] = pop
        dfs.append(pop_df)
    return pd.concat(dfs, ignore_index=True)

genotype_df = generate_genotypes(freq_df, F, n_per_deme)

# ── MAF, flip, rename, sort ───────────────────────────────────────────────────
def compute_maf(df, snp_cols):
    allele_counts = (df[snp_cols] == -1) * 2 + (df[snp_cols] == 0)
    return allele_counts.mean(axis=0) / 2

def maf_prefix(m):
    if m < 0.01:  return "VR"
    elif m < 0.05: return "R"
    else:          return "C"

snp_cols = [c for c in genotype_df.columns if c != 'population']
maf      = compute_maf(genotype_df, snp_cols)

cols_to_flip = maf[maf > 0.5].index
genotype_df[cols_to_flip] = genotype_df[cols_to_flip] * -1
maf = compute_maf(genotype_df, snp_cols)    # recompute after flip

sorted_cols = maf.sort_values().index
new_columns = {
    col: f"{maf_prefix(maf[col])}_{maf[col]:.2f}_snp_{i+1}"
    for i, col in enumerate(sorted_cols)
}
genotype_df = genotype_df[list(sorted_cols) + ['population']].rename(columns=new_columns)

snp_cols = [col for col in genotype_df.columns if col != 'population']

G_f    = genotype_df[snp_cols]
std    = G_f.std(axis=0); std[std == 0] = 1
G_std  = (G_f - G_f.mean(axis=0)) / std
pca    = PCA(n_components=2)
coords = pca.fit_transform(G_std)

deme_colors = build_deme_colors(k)
sample_demes = np.repeat(np.arange(k*k), n_per_deme)
hap_colors   = [deme_colors[d] for d in sample_demes]

fig_pca  = visualize_pca(hap_colors, coords)
visualize_multiple_plots([fig_pca], figsize_each=(2, 2))

number_of_genes = 4

# --- 1. Transpose: SNPs as rows, individuals as columns ---
snp_cols = [c for c in genotype_df.columns if c != 'population']
df_snps = genotype_df[snp_cols].T  # shape: (2000, 10000)

# --- 2. Compute pairwise distances between SNPs ---
# Correlation-based distance: SNPs with similar population patterns = small distance
distances = pdist(df_snps.values, metric='correlation')

# --- 3. Hierarchical clustering ---
Z = linkage(distances, method='ward')

# --- 4. Cut into 4 clusters ---
labels = fcluster(Z, t=number_of_genes, criterion='maxclust')

# --- 5. Assign cluster back to SNP names ---
snp_cluster_map = pd.Series(labels, index=snp_cols, name='gene_cluster')

# --- 6. Optional: visualize the dendrogram ---
plt.figure(figsize=(12, 4))
dendrogram(Z, no_labels=True, truncate_mode='lastp', p=50)

# Automatically find the cut height for 4 clusters
# Z[-3, 2] is the height of the 3rd-to-last merge (cutting here gives 4 clusters)
cut_height = (Z[-3, 2] + Z[-4, 2]) / 2  # midpoint between the 3rd and 4th last merges
plt.axhline(y=cut_height, color='r', linestyle='--', label=f'cut @ {cut_height:.2f}')
plt.title("SNP Dendrogram")
plt.legend()
plt.show()

gene_dfs = {}
for gene_id in range(1, number_of_genes+1):
    gene_snps = snp_cluster_map[snp_cluster_map == gene_id].index.tolist()
    gene_dfs[f'gene_{gene_id}'] = genotype_df[gene_snps + ['population']]
    print(f"gene_{gene_id}: {len(gene_snps)} SNPs")

from pathlib import Path

out_dir = Path('data/genes')
out_dir.mkdir(exist_ok=True)

for name, df in gene_dfs.items():
    df.to_parquet(out_dir / f'{name}.parquet')

# also persist the assignment + linkage so the split is reproducible
snp_cluster_map.to_frame().to_parquet(out_dir / 'snp_cluster_map.parquet')
np.save(out_dir / 'linkage_Z.npy', Z)

# Load params

# Load data

In [ ]:
genotype_df

# Select n genes based on correlation

# Vizualize Genes position

In [ ]:
# plot scatter plot all populations

In [ ]:
test = gene_dfs['gene_1']
test = test.drop(columns=["population"])
snp_data = test.values
n_samples = snp_data.shape[0]
pca = PCA(n_components=3)
scaler = MinMaxScaler(feature_range=(0, 255))
n_samples = snp_data.shape[0]
n_components = min(3, n_samples, snp_data.shape[1])
pca = PCA(n_components=n_components)
reduced = pca.fit_transform(snp_data)  # shape: (n_individuals, 3)
rgb = scaler.fit_transform(reduced).astype(np.uint8)  # shape: (n_individuals, 3)
colors = rgb / 255.0  # matplotlib expects float [0, 1]

fig, ax = plt.subplots(figsize=(8, 6))
ax.scatter(reduced[:, 0], reduced[:, 1], c=colors, s=10, alpha=0.8)

ax.set_xlabel("PC1")
ax.set_ylabel("PC2")
ax.set_title("PCA – 2D scatter colored by PC1/PC2/PC3 → RGB")
plt.tight_layout()
plt.show()

In [ ]:
%matplotlib widget
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import numpy as np
import json
import os

K = 10
selections = {}
selected_color = [[255, 0, 0]]
SAVE_PATH = 'selection_targets.json'

COLORS = [
    ('red',     [255, 0,   0]),
    ('green',   [0,   255, 0]),
    ('blue',    [0,   0, 255]),
    ('yellow',  [255, 255, 0]),
    ('cyan',    [0,   255, 255]),
    ('magenta', [255, 0,   255]),
    ('orange',  [255, 128, 0]),
    ('white',   [255, 255, 255]),
    ('black',   [0,   0,   0]),
]

QUAD_TO_GENE = {
    (0, 1): 'gene_1',
    (1, 1): 'gene_2',
    (0, 0): 'gene_3',
    (1, 0): 'gene_4',
}

fig = plt.figure(figsize=(15, 8))
ax_grid = fig.add_axes([0.02, 0.05, 0.58, 0.90])
ax_ctrl = fig.add_axes([0.63, 0.05, 0.35, 0.90])
ax_ctrl.set_xlim(0, 1); ax_ctrl.set_ylim(0, 1)
ax_ctrl.axis('off')

# store button regions in AXIS coords for reliable hit detection
swatch_boxes = []
BUTTONS = {}   # name → (x0, y0, x1, y1)

def draw_grid():
    ax_grid.clear()
    ax_grid.set_xlim(0, K); ax_grid.set_ylim(0, K)
    ax_grid.set_aspect('equal'); ax_grid.axis('off')
    ax_grid.set_title('click a quadrant inside a deme to paint / unpaint', fontsize=9)
    for m in range(K * K):
        pop = m + 1
        r, c = m // K, m % K
        cx, cy = c, K - 1 - r
        for (sc, sr), gene in QUAD_TO_GENE.items():
            x = cx + sc * 0.5
            y = cy + sr * 0.5
            rgb = selections.get(pop, {}).get(gene)
            color = np.array(rgb) / 255.0 if rgb is not None else [0.88, 0.88, 0.88]
            ax_grid.add_patch(patches.Rectangle(
                (x, y), 0.5, 0.5, color=color, ec='white', lw=0.5))
        ax_grid.add_patch(patches.Rectangle(
            (cx, cy), 1, 1, fill=False, ec='#555', lw=1.2))
        ax_grid.text(cx + 0.06, cy + 0.06, str(pop),
                     fontsize=5, color='#666', va='bottom')
    fig.canvas.draw_idle()

def draw_controls():
    ax_ctrl.clear()
    ax_ctrl.set_xlim(0, 1); ax_ctrl.set_ylim(0, 1)
    ax_ctrl.axis('off')
    swatch_boxes.clear()
    BUTTONS.clear()

    ax_ctrl.text(0.05, 0.97, 'target color', fontsize=9, color='gray', va='top')

    # swatches — 3 per row
    sw_w, sw_h, sw_gx, sw_gy = 0.26, 0.075, 0.03, 0.02
    for i, (name, rgb) in enumerate(COLORS):
        col, row = i % 3, i // 3
        x = 0.05 + col * (sw_w + sw_gx)
        y = 0.89 - row * (sw_h + sw_gy)
        active = (rgb == selected_color[0])
        ax_ctrl.add_patch(patches.Rectangle(
            (x, y), sw_w, sw_h,
            color=np.array(rgb) / 255.0,
            ec='black' if active else '#bbb',
            lw=2.5 if active else 0.5, zorder=2))
        if active:
            tc = 'white' if sum(rgb) < 400 else 'black'
            ax_ctrl.text(x + sw_w/2, y + sw_h/2, '✓',
                         ha='center', va='center', fontsize=10, color=tc, zorder=3)
        swatch_boxes.append((x, y, x + sw_w, y + sw_h, rgb))

    # selected preview
    ax_ctrl.text(0.05, 0.52, 'selected:', fontsize=8, color='gray', va='top')
    ax_ctrl.add_patch(patches.Rectangle(
        (0.05, 0.42), 0.90, 0.08,
        color=np.array(selected_color[0]) / 255.0, ec='black', lw=0.8))

    # position key
    ax_ctrl.text(0.05, 0.40, 'position key:', fontsize=8, color='gray', va='top')
    for (sc, sr), gene in QUAD_TO_GENE.items():
        x = 0.05 + sc * 0.47
        y = 0.25 + sr * 0.09
        ax_ctrl.add_patch(patches.Rectangle(
            (x, y), 0.42, 0.08, color='#ddd', ec='#999', lw=0.8))
        ax_ctrl.text(x + 0.21, y + 0.04, gene,
                     ha='center', va='center', fontsize=7.5)

    # buttons
    def draw_btn(name, label, y0, fc, ec):
        box = (0.05, y0, 0.95, y0 + 0.07)
        BUTTONS[name] = box
        x0, y0b, x1, y1 = box
        ax_ctrl.add_patch(patches.FancyBboxPatch(
            (x0, y0b), x1 - x0, y1 - y0b,
            boxstyle='round,pad=0.01', fc=fc, ec=ec, lw=0.8))
        ax_ctrl.text((x0 + x1) / 2, (y0b + y1) / 2, label,
                     ha='center', va='center', fontsize=8)

    draw_btn('save', 'save selection_targets.json', 0.14, '#e8f4e8', '#8a8')
    draw_btn('print', 'print fitness_targets',       0.06, '#e8e8e8', '#888')
    draw_btn('clear', 'clear all', 0.00, '#fde8e8', '#c88')

    fig.canvas.draw_idle()

def selections_to_fitness_targets():
    """Convert selections dict → fitness_targets dict."""
    ft = {}
    for pop in sorted(selections):
        for gene in ['gene_1', 'gene_2', 'gene_3', 'gene_4']:
            if gene in selections.get(pop, {}):
                ft[(pop, gene)] = tuple(selections[pop][gene])
    return ft

def print_targets():
    lines = ['fitness_targets = {']
    for pop in sorted(selections):
        for gene in ['gene_1', 'gene_2', 'gene_3', 'gene_4']:
            if gene in selections.get(pop, {}):
                r, g, b = selections[pop][gene]
                lines.append(f"    ({pop}, '{gene}'): ({r}, {g}, {b}),")
    lines.append('}')
    print('\n'.join(lines))

def save_json():
    out = {}
    for pop in sorted(selections):
        out[str(pop)] = {
            gene: selections[pop][gene]
            for gene in selections[pop]
        }
    with open(SAVE_PATH, 'w') as f:
        json.dump(out, f, indent=2)
    print(f"Saved {len(out)} demes → {SAVE_PATH}")
    print_targets()

def load_json(path=SAVE_PATH):
    """Load a previously saved file back into selections and redraw."""
    if not os.path.exists(path):
        print(f"File not found: {path}")
        return {}
    with open(path) as f:
        data = json.load(f)
    selections.clear()
    for pop_str, genes in data.items():
        selections[int(pop_str)] = {g: v for g, v in genes.items()}
    draw_grid()
    ft = selections_to_fitness_targets()
    print(f"Loaded {len(selections)} demes from {path}")
    return ft

def _in_box(x, y, box):
    x0, y0, x1, y1 = box
    return x0 <= x <= x1 and y0 <= y <= y1

def on_click(event):
    if event.inaxes == ax_grid and event.xdata is not None:
        c = int(event.xdata)
        r = int(event.ydata)
        if not (0 <= c < K and 0 <= r < K):
            return
        pop = (K - 1 - r) * K + c + 1
        sc = 1 if (event.xdata - c) >= 0.5 else 0
        sr = 1 if (event.ydata - r) >= 0.5 else 0
        gene = QUAD_TO_GENE[(sc, sr)]
        cur = selections.get(pop, {}).get(gene)
        if cur == selected_color[0]:
            del selections[pop][gene]
            if not selections[pop]:
                del selections[pop]
        else:
            selections.setdefault(pop, {})[gene] = list(selected_color[0])
        print(f"pop {pop} · {gene} → {selected_color[0]}")
        draw_grid()

    elif event.inaxes == ax_ctrl and event.xdata is not None:
        x, y = event.xdata, event.ydata

        for (x0, y0, x1, y1, rgb) in swatch_boxes:
            if x0 <= x <= x1 and y0 <= y <= y1:
                selected_color[0] = list(rgb)
                draw_controls()
                return

        for name, box in BUTTONS.items():
            if _in_box(x, y, box):
                if name == 'save':  save_json()
                elif name == 'print': print_targets()
                elif name == 'clear':
                    selections.clear()
                    draw_grid()
                return

fig.canvas.mpl_connect('button_press_event', on_click)
draw_grid()
draw_controls()
plt.show()

In [ ]:
# Load saved selections as fitness_targets
fitness_targets = load_json('selection_targets.json')
print(fitness_targets)

In [ ]:
plt.close('all')

In [ ]:
"""
Forward-in-time population genetics simulator
==============================================
Pipeline
--------
1. phase_to_haplotypes()   — convert diploid -1/0/1 matrix → two haplotype arrays
2. fit_gen0()              — fit one PCA per (pop, gene), store for all generations
3. compute_state()         — project haplotypes → diploid → PCA → RGB → fitness
4. reproduce()             — selection + recombination → next generation haplotypes
5. haplotypes_to_diploid() — convert haplotypes back to gene_dfs for grid/PCA
6. plot_gene_grid()        — unchanged visual

Assumptions
-----------
- Genes are UNLINKED: each gene recombines independently (realistic for genes on
  different chromosomes or far apart).
- SNP order within a gene is randomised at initialisation (no physical map).
- Het sites (0) are randomly phased once at initialisation.
- Single crossover point drawn uniformly per gene per individual per generation.
- Population size is fixed (N individuals per deme, N preserved by resampling).
- No migration between demes.
"""

from __future__ import annotations
import numpy as np
import pandas as pd
import matplotlib.patches as patches
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from typing import Optional


# ============================================================
# 1. PHASING  —  diploid (-1/0/1) → two haplotype arrays
# ============================================================

def phase_to_haplotypes(gene_dfs: dict) -> dict:
    """
    Convert each gene_df (diploid -1/0/1 + population column) into a dict of
    haplotype arrays per population.

    Encoding convention:
        -1  → hom ref   → hap_a = 0, hap_b = 0
         0  → het       → hap_a = 0, hap_b = 1  (randomly phased)
         1  → hom alt   → hap_a = 1, hap_b = 1

    Returns
    -------
    haplotypes : dict
        haplotypes[gene_name][pop] = {
            'a': np.ndarray (N, L)  int8   haplotype A
            'b': np.ndarray (N, L)  int8   haplotype B
            'snp_order': list[str]         randomised SNP column order
        }
    """
    haplotypes: dict = {}

    for gene_name, df in gene_dfs.items():
        haplotypes[gene_name] = {}
        snp_cols = [c for c in df.columns if c != 'population']

        # Randomise SNP order within gene (no physical map)
        rng_order = np.random.permutation(len(snp_cols))
        snp_order = [snp_cols[i] for i in rng_order]

        for pop in sorted(df['population'].unique()):
            subset = df[df['population'] == pop][snp_order].values  # (N, L)
            N, L = subset.shape

            hap_a = np.zeros((N, L), dtype=np.int8)
            hap_b = np.zeros((N, L), dtype=np.int8)

            # hom ref (-1) → 0 | 0  (already zeros)
            # hom alt ( 1) → 1 | 1
            hom_alt = subset == 1
            hap_a[hom_alt] = 1
            hap_b[hom_alt] = 1

            # het (0) → randomly assign which haplotype carries the alt
            het = subset == 0
            het_idx = np.argwhere(het)         # (n_het, 2)
            rand_flip = np.random.randint(0, 2, size=len(het_idx))
            for (i, j), flip in zip(het_idx, rand_flip):
                hap_a[i, j] = flip
                hap_b[i, j] = 1 - flip

            haplotypes[gene_name][pop] = {
                'a': hap_a,
                'b': hap_b,
                'snp_order': snp_order,
            }

    return haplotypes


# ============================================================
# 2.  haplotypes → diploid gene_dfs  (for PCA / grid)
# ============================================================

def haplotypes_to_diploid(haplotypes: dict) -> dict:
    """
    Reconstruct gene_dfs (same format as input) from haplotype arrays.
    Diploid encoding: hap_a + hap_b - 1  →  {-1, 0, 1}
    """
    gene_dfs: dict = {}

    for gene_name, pop_dict in haplotypes.items():
        frames = []
        for pop, data in pop_dict.items():
            diploid = (data['a'].astype(np.int16) +
                       data['b'].astype(np.int16) - 1)  # (N, L)
            df = pd.DataFrame(diploid, columns=data['snp_order'])
            df['population'] = pop
            frames.append(df)
        gene_dfs[gene_name] = pd.concat(frames, ignore_index=True)

    return gene_dfs


# ============================================================
# 2b. gene_dfs → single genotype_df  (mirrors your original input)
# ============================================================

def gene_dfs_to_genotype_df(gene_dfs: dict) -> pd.DataFrame:
    """
    Reconstruct a single genotype_df from gene_dfs — the exact same format
    as your original input (one row per individual, all SNP columns from all
    genes merged, plus a 'population' column).

    Individuals are matched across genes by position within each population
    block: all pop-1 individuals first, then pop-2, etc.

    Usage
    -----
    final_gene_dfs = haplotypes_to_diploid(haplotypes)
    final_genotype = gene_dfs_to_genotype_df(final_gene_dfs)

    # Or in one step:
    final_genotype = haplotypes_to_genotype_df(haplotypes)
    """
    gene_names = list(gene_dfs.keys())
    first = gene_dfs[gene_names[0]]
    populations = sorted(first['population'].unique())

    pop_frames = []
    for pop in populations:
        gene_slices = []
        for gene_name in gene_names:
            df = gene_dfs[gene_name]
            subset = (df[df['population'] == pop]
                      .drop(columns=['population'])
                      .reset_index(drop=True))
            gene_slices.append(subset)
        merged = pd.concat(gene_slices, axis=1)
        merged['population'] = pop
        pop_frames.append(merged)

    genotype_df = pd.concat(pop_frames, ignore_index=True)
    cols = [c for c in genotype_df.columns if c != 'population'] + ['population']
    return genotype_df[cols]


def haplotypes_to_genotype_df(haplotypes: dict) -> pd.DataFrame:
    """Convenience wrapper: haplotypes → genotype_df in one call."""
    return gene_dfs_to_genotype_df(haplotypes_to_diploid(haplotypes))


# ============================================================
# 3.  Fit gen-0 PCA  (call once, reuse every generation)
# ============================================================

def fit_gen0(gene_dfs: dict, fitness_targets: dict, gamma: float = 2.5):
    """
    Fit one PCA per (pop, gene) on generation-0 diploid data.
    Store projector + gen-0 rescaling params so future generations
    map to the same colour space.

    Parameters
    ----------
    fitness_targets : dict
        {(pop, gene_name): (R, G, B)}  values in [0, 255]

    Returns
    -------
    gen0_pcas : dict  keyed by (pop, gene_name)
    state     : GenerationState dict  (see compute_state)
    """
    gene_names = list(gene_dfs.keys())
    populations = sorted(gene_dfs[gene_names[0]]['population'].unique())

    gen0_pcas: dict = {}
    state = _compute_state_from_dfs(gene_dfs, gen0_pcas, fitness_targets,
                                    gamma, fitting=True)
    return gen0_pcas, state


def _compute_state_from_dfs(gene_dfs, gen0_pcas, fitness_targets,
                             gamma, fitting=False):
    """Internal: shared logic for fit_gen0 and compute_state."""
    gene_names = list(gene_dfs.keys())
    populations = sorted(gene_dfs[gene_names[0]]['population'].unique())
    state: dict = {}

    for pop in populations:
        state[pop] = {}
        for gene_name in gene_names:
            df = gene_dfs[gene_name]
            subset = df[df['population'] == pop].drop(columns=['population'])
            snp_data = subset.values.astype(float)
            n_samples, n_features = snp_data.shape
            n_components = min(3, n_samples, n_features)

            if fitting:
                pca = PCA(n_components=n_components)
                reduced = pca.fit_transform(snp_data)
                if reduced.shape[1] < 3:
                    pad = np.zeros((reduced.shape[0], 3 - reduced.shape[1]))
                    reduced = np.hstack([reduced, pad])
                ch_min = reduced.min(axis=0)
                ch_range = np.where(
                    reduced.max(axis=0) - ch_min == 0, 1.0,
                    reduced.max(axis=0) - ch_min
                )
                gen0_pcas[(pop, gene_name)] = {
                    'pca': pca,
                    'ch_min': ch_min,
                    'ch_range': ch_range,
                }
            else:
                proj = gen0_pcas[(pop, gene_name)]
                reduced = proj['pca'].transform(snp_data)
                if reduced.shape[1] < 3:
                    pad = np.zeros((reduced.shape[0], 3 - reduced.shape[1]))
                    reduced = np.hstack([reduced, pad])
                ch_min = proj['ch_min']
                ch_range = proj['ch_range']

            rgb_float = ((reduced - ch_min) / ch_range).clip(0, 1)
            rgb_uint8 = (rgb_float * 255).astype(np.uint8)

            target = fitness_targets.get((pop, gene_name))
            if target is not None:
                target_arr = np.array(target, dtype=np.float32)
                distances = np.linalg.norm(
                    rgb_uint8.astype(np.float32) - target_arr, axis=1)
                fitness = 1.0 - distances / np.sqrt(3 * 255 ** 2)
            else:
                fitness = None

            state[pop][gene_name] = {
                'rgb':     rgb_uint8,
                'display': (rgb_float ** gamma).clip(0, 1),
                'fitness': fitness,
                'reduced': reduced,
            }

    return state


def compute_state(gene_dfs: dict, gen0_pcas: dict,
                  fitness_targets: dict, gamma: float = 2.5) -> dict:
    """Project current generation onto gen-0 PCA space, compute fitness."""
    return _compute_state_from_dfs(gene_dfs, gen0_pcas, fitness_targets, gamma)


# ============================================================
# 4.  REPRODUCTION  —  selection + recombination
# ============================================================

def reproduce(haplotypes: dict, state: dict,
              recombination_rate: float = 1.0,
              gene_order: list = None,
              neighbors: dict = None,
              migration_rate: float = 0.05) -> dict:
    """
    Produce the next generation via fitness-weighted selection + single
    chromosomal crossover + stepping-stone migration.

    Chromosome model
    ----------------
    Genes are laid out in a fixed linear order (default: gene_1→gene_2→gene_3→gene_4).
    One crossover point is drawn per gamete along the entire chromosome.
    Everything left of the crossover comes from haplotype A, right from B.

    Migration model (stepping-stone)
    ---------------------------------
    Each deme's new generation has two sources:
      - Local offspring  : (1 - migration_rate) * N individuals, parents drawn
                           from the focal deme with fitness weights.
      - Migrant offspring: migration_rate * N individuals total, split equally
                           across neighboring demes. Parents are drawn from the
                           neighbor deme but fitness-weighted by the *receiving*
                           deme's fitness targets (migrants must prove themselves).

    Population size N is exactly preserved in every deme every generation.
    Migration is symmetric: all neighbors contribute equally.

    Parameters
    ----------
    haplotypes       : current generation haplotypes
    state            : current GenerationState (fitness scores)
    recombination_rate : probability of a crossover per gamete [0, 1]
    gene_order       : list of gene names in chromosome order
    neighbors        : dict mapping pop_id (0-indexed) → list of neighbor pop_ids
                       from create_grid(). If None, no migration occurs.
    migration_rate   : fraction of each deme replaced by migrants (default 0.05)

    Returns
    -------
    new_haplotypes : dict  same structure as input haplotypes
    """
    if gene_order is None:
        gene_order = ['gene_1', 'gene_2', 'gene_3', 'gene_4']

    gene_names = list(haplotypes.keys())
    populations = sorted(next(iter(haplotypes.values())).keys())
    N = haplotypes[gene_order[0]][populations[0]]['a'].shape[0]

    # --- chromosome coordinate map ---
    gene_lengths = {g: haplotypes[g][populations[0]]['a'].shape[1]
                    for g in gene_order}
    chrom_len = sum(gene_lengths.values())
    gene_starts = {}
    pos = 0
    for g in gene_order:
        gene_starts[g] = pos
        pos += gene_lengths[g]

    # --- fitness weights for a given pop, evaluated in that pop's context ---
    def get_weights(pop):
        all_fitness = []
        for g in gene_order:
            f = state[pop][g]['fitness']
            if f is not None:
                all_fitness.append(f)
        if not all_fitness:
            return np.ones(N) / N
        mean_f = np.mean(all_fitness, axis=0) + 1e-6
        return mean_f / mean_f.sum()

    # --- draw gametes from a source pop, weighted by receiving pop's fitness ---
    def make_gametes(source_pop, receiving_pop, n):
        """
        Draw n gametes from source_pop's haplotypes, but use receiving_pop's
        fitness weights (migrants are selected by the destination environment).
        """
        # Use receiving deme's fitness to weight parents from source deme.
        # If source pop size differs, re-weight by source individuals' fitness
        # evaluated in the receiving context — approximated by receiving weights
        # recomputed on source individuals' RGB (not currently stored per-source).
        # Simpler and standard: use receiving deme's own weight vector shape,
        # but draw parents from source. Since N is fixed, weights length matches.
        w = get_weights(receiving_pop)  # weights of receiving pop individuals
        # We can't directly apply receiving weights to source individuals
        # (different people). Use uniform selection from source instead —
        # natural selection acts on offspring in next generation's state update.
        # This is the standard "migrant pool" model.
        w_source = get_weights(source_pop)
        parent_idx = np.random.choice(N, size=n, replace=True, p=w_source)

        gamete_genes = {g: [] for g in gene_order}
        for pid in parent_idx:
            if recombination_rate > 0 and np.random.rand() < recombination_rate:
                xover = np.random.randint(0, chrom_len)
            else:
                xover = None

            left_hap, right_hap = ('a', 'b') if np.random.rand() < 0.5 else ('b', 'a')

            for g in gene_order:
                a = haplotypes[g][source_pop]['a'][pid]
                b = haplotypes[g][source_pop]['b'][pid]
                hap = {'a': a, 'b': b}
                L_g = gene_lengths[g]
                g_start = gene_starts[g]
                g_end = g_start + L_g

                if xover is None:
                    seg = hap[left_hap].copy()
                elif xover <= g_start:
                    seg = hap[right_hap].copy()
                elif xover >= g_end:
                    seg = hap[left_hap].copy()
                else:
                    local_xover = xover - g_start
                    seg = np.empty(L_g, dtype=np.int8)
                    seg[:local_xover] = hap[left_hap][:local_xover]
                    seg[local_xover:] = hap[right_hap][local_xover:]

                gamete_genes[g].append(seg)

        return {g: np.array(segs, dtype=np.int8) for g, segs in gamete_genes.items()}

    # --- produce offspring for every deme ---
    # Accumulate gamete pairs (hap_a, hap_b) per gene per pop
    new_haplotypes: dict = {g: {} for g in gene_names}

    for pop in populations:
        # pop is 1-indexed (from your gene_dfs), neighbors dict is 0-indexed
        pop_0 = pop - 1
        nbr_list = neighbors.get(pop_0, []) if neighbors is not None else []
        n_nbrs = len(nbr_list)

        # --- how many offspring from each source ---
        if n_nbrs == 0 or neighbors is None or migration_rate == 0:
            n_local = N
            migrant_counts = {}
        else:
            n_migrants_total = round(N * migration_rate)
            n_local = N - n_migrants_total
            # Distribute migrants as evenly as possible across neighbors
            base = n_migrants_total // n_nbrs
            remainder = n_migrants_total % n_nbrs
            migrant_counts = {}
            for i, nbr in enumerate(nbr_list):
                migrant_counts[nbr + 1] = base + (1 if i < remainder else 0)  # back to 1-indexed

        # --- local gametes ---
        ga_local = make_gametes(pop, pop, n_local)
        gb_local = make_gametes(pop, pop, n_local)

        # --- migrant gametes from each neighbor ---
        ga_parts = {g: [ga_local[g]] for g in gene_order}
        gb_parts = {g: [gb_local[g]] for g in gene_order}

        for src_pop_1idx, n_mig in migrant_counts.items():
            if n_mig == 0:
                continue
            ga_mig = make_gametes(src_pop_1idx, pop, n_mig)
            gb_mig = make_gametes(src_pop_1idx, pop, n_mig)
            for g in gene_order:
                ga_parts[g].append(ga_mig[g])
                gb_parts[g].append(gb_mig[g])

        for g in gene_order:
            new_haplotypes[g][pop] = {
                'a':         np.vstack(ga_parts[g]),   # (N, L)
                'b':         np.vstack(gb_parts[g]),
                'snp_order': haplotypes[g][pop]['snp_order'],
            }

    return new_haplotypes


# ============================================================
# 5.  LD MEASUREMENT  —  r² between SNP pairs within a gene
# ============================================================

def compute_ld(haplotypes: dict, gene_name: str, pop: int,
               max_snps: int = 100) -> np.ndarray:
    """
    Compute pairwise r² (standard LD measure) between SNPs for one (pop, gene).
    Uses haplotypes directly (phased data) for accurate LD estimation.

    Returns r2 matrix of shape (L, L) where L = min(n_snps, max_snps).
    """
    data = haplotypes[gene_name][pop]
    # Stack both haplotypes — each row is one chromosome
    h = np.vstack([data['a'], data['b']]).astype(float)  # (2N, L)

    L = h.shape[1]
    if L > max_snps:
        idx = np.random.choice(L, max_snps, replace=False)
        idx.sort()
        h = h[:, idx]
        L = max_snps

    # Allele frequencies
    p = h.mean(axis=0)          # freq of alt allele  (L,)
    q = 1 - p

    # Centre
    h_c = h - p

    # r² = (cov(i,j))² / (var(i) * var(j))
    cov = (h_c.T @ h_c) / h.shape[0]
    var = np.diag(cov)
    denom = np.outer(var, var)
    denom = np.where(denom == 0, 1e-10, denom)
    r2 = (cov ** 2) / denom
    np.fill_diagonal(r2, 1.0)
    return r2


def plot_ld(haplotypes: dict, gene_name: str, pop: int,
            max_snps: int = 100, title: str = ''):
    r2 = compute_ld(haplotypes, gene_name, pop, max_snps)
    fig, ax = plt.subplots(figsize=(6, 5))
    im = ax.imshow(r2, cmap='Reds', vmin=0, vmax=1, aspect='auto')
    plt.colorbar(im, ax=ax, label='r²')
    ax.set_title(title or f'LD (r²) — {gene_name}, pop {pop}', fontsize=11)
    ax.set_xlabel('SNP index')
    ax.set_ylabel('SNP index')
    plt.tight_layout()
    plt.show()



# ============================================================
# 5b. GENOME-WIDE LD  —  heatmap + decay curve (paper figures)
# ============================================================

def _compute_r2_matrix(h: np.ndarray) -> np.ndarray:
    """Compute pairwise r² from a (2N, L) phased haplotype matrix."""
    p = h.mean(axis=0)
    h_c = h - p
    cov = (h_c.T @ h_c) / h.shape[0]
    var = np.diag(cov)
    denom = np.outer(var, var)
    denom = np.where(denom == 0, 1e-10, denom)
    r2 = (cov ** 2) / denom
    np.fill_diagonal(r2, 1.0)
    return r2


def plot_genome_wide_ld(
    haplotypes: dict,
    pop: int,
    gene_order: list = None,
    max_snps_per_gene: int = 100,
    generation: int = None,
    figsize: tuple = (12, 10),
):
    """
    Figure 2 + 3 for a paper: genome-wide LD heatmap and LD decay curve.

    Parameters
    ----------
    haplotypes        : haplotype dict from any generation
    pop               : population to analyse (1-indexed)
    gene_order        : chromosome order of genes (default gene_1→4)
    max_snps_per_gene : subsample to this many SNPs per gene (default 100)
                        → 400 total SNPs in the genome-wide heatmap
    generation        : generation number for plot titles (optional)
    figsize           : figure size for the combined figure

    Returns
    -------
    r2_full  : np.ndarray  full genome-wide r² matrix (400×400)
    r2_decay : pd.DataFrame  columns: distance (SNP units), mean_r2, gene_pair
    """
    if gene_order is None:
        gene_order = ['gene_1', 'gene_2', 'gene_3', 'gene_4']

    gen_label = f'gen {generation}' if generation is not None else ''

    # --- 1. Build subsampled genome-wide haplotype matrix ---
    # Keep a fixed random subsample per gene (reproducible within a run)
    gene_blocks = []       # list of (gene_name, h_sub, snp_indices)
    gene_boundaries = []   # (start, end) in the concatenated matrix
    pos = 0

    for gene_name in gene_order:
        data = haplotypes[gene_name][pop]
        h = np.vstack([data['a'], data['b']]).astype(float)  # (2N, L)
        L = h.shape[1]

        if L > max_snps_per_gene:
            idx = np.sort(np.random.choice(L, max_snps_per_gene, replace=False))
            h_sub = h[:, idx]
        else:
            idx = np.arange(L)
            h_sub = h

        n_snps = h_sub.shape[1]
        gene_blocks.append((gene_name, h_sub, idx))
        gene_boundaries.append((pos, pos + n_snps))
        pos += n_snps

    total_snps = pos
    h_genome = np.hstack([b[1] for b in gene_blocks])   # (2N, total_snps)

    # --- 2. Compute genome-wide r² ---
    r2_full = _compute_r2_matrix(h_genome)

    # --- 3. Figure A: genome-wide heatmap ---
    fig, axes = plt.subplots(1, 2, figsize=figsize)

    ax = axes[0]
    im = ax.imshow(r2_full, cmap='Reds', vmin=0, vmax=1,
                   aspect='auto', interpolation='nearest')
    plt.colorbar(im, ax=ax, label='r²', fraction=0.046, pad=0.04)

    # Draw gene block boundaries
    colors_key = ['#4444cc', '#cc8844', '#44aa44', '#cc4444']
    for i, (gene_name, (start, end)) in enumerate(zip(gene_order, gene_boundaries)):
        mid = (start + end) / 2
        # Box outline on heatmap
        rect = plt.Rectangle((start - 0.5, start - 0.5),
                               end - start, end - start,
                               fill=False, edgecolor=colors_key[i],
                               linewidth=2, linestyle='--')
        ax.add_patch(rect)
        ax.text(mid, -2, gene_name, ha='center', va='bottom',
                fontsize=8, color=colors_key[i], fontweight='bold',
                clip_on=False)
        ax.text(-2, mid, gene_name, ha='right', va='center',
                fontsize=8, color=colors_key[i], fontweight='bold',
                clip_on=False)

    ax.set_xlim(-0.5, total_snps - 0.5)
    ax.set_ylim(total_snps - 0.5, -0.5)
    ax.set_xlabel('SNP index (chromosome order)')
    ax.set_ylabel('SNP index (chromosome order)')
    title = f'Genome-wide LD (r²) — pop {pop}'
    if gen_label:
        title += f', {gen_label}'
    ax.set_title(title, fontsize=11)

    # --- 4. Figure B: LD decay curve ---
    ax2 = axes[1]

    # Compute r² vs distance for each gene (within-gene) and between genes
    decay_records = []

    # Within-gene decay
    for i, (gene_name, (start, end)) in enumerate(zip(gene_order, gene_boundaries)):
        r2_gene = r2_full[start:end, start:end]
        n = end - start
        for d in range(1, n):
            # All pairs at distance d (in subsampled SNP units)
            diag_vals = [r2_gene[j, j + d] for j in range(n - d)]
            decay_records.append({
                'distance':  d,
                'mean_r2':   np.mean(diag_vals),
                'pair_type': f'within {gene_name}',
                'color':     colors_key[i],
            })

    # Between-gene decay (use distance from end of gene i to start of gene j)
    for i in range(len(gene_order)):
        for j in range(i + 1, len(gene_order)):
            s_i, e_i = gene_boundaries[i]
            s_j, e_j = gene_boundaries[j]
            r2_block = r2_full[s_i:e_i, s_j:e_j]
            mean_r2 = r2_block.mean()
            # Representative distance = midpoint separation
            mid_dist = int((s_j + e_j) / 2) - int((s_i + e_i) / 2)
            decay_records.append({
                'distance':  mid_dist,
                'mean_r2':   mean_r2,
                'pair_type': f'{gene_order[i]}–{gene_order[j]}',
                'color':     '#888888',
            })

    decay_df = pd.DataFrame(decay_records)

    # Plot within-gene curves
    for i, gene_name in enumerate(gene_order):
        sub = decay_df[decay_df['pair_type'] == f'within {gene_name}']
        ax2.plot(sub['distance'], sub['mean_r2'],
                 color=colors_key[i], linewidth=1.5,
                 label=f'within {gene_name}', alpha=0.85)

    # Plot between-gene points
    between = decay_df[~decay_df['pair_type'].str.startswith('within')]
    ax2.scatter(between['distance'], between['mean_r2'],
                color='#444444', s=60, zorder=5,
                label='between genes', marker='D')
    for _, row in between.iterrows():
        ax2.annotate(row['pair_type'],
                     xy=(row['distance'], row['mean_r2']),
                     xytext=(5, 5), textcoords='offset points',
                     fontsize=6.5, color='#444444')

    ax2.set_xlabel('Distance (SNP units along chromosome)')
    ax2.set_ylabel('Mean r²')
    title2 = f'LD decay — pop {pop}'
    if gen_label:
        title2 += f', {gen_label}'
    ax2.set_title(title2, fontsize=11)
    ax2.legend(fontsize=8, loc='upper right')
    ax2.set_ylim(0, 1)
    ax2.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()

    return r2_full, decay_df


def compare_ld_generations(
    haplotypes_list: list,
    generations: list,
    pop: int,
    gene_name: str,
    gene_order: list = None,
    max_snps: int = 100,
):
    """
    Figure 1 for a paper: per-gene LD heatmaps side by side across generations.
    Pass e.g. [haplotypes_gen0, haplotypes_gen10, haplotypes_gen30] with
    generations=[0, 10, 30] to see LD buildup over time.

    Parameters
    ----------
    haplotypes_list : list of haplotype dicts (one per generation to compare)
    generations     : list of generation numbers (for titles)
    pop             : population to analyse
    gene_name       : which gene to show
    max_snps        : subsample to this many SNPs
    """
    n = len(haplotypes_list)
    fig, axes = plt.subplots(1, n, figsize=(5 * n, 4.5))
    if n == 1:
        axes = [axes]

    for ax, haps, gen in zip(axes, haplotypes_list, generations):
        r2 = compute_ld(haps, gene_name, pop, max_snps=max_snps)
        im = ax.imshow(r2, cmap='Reds', vmin=0, vmax=1, aspect='auto')
        ax.set_title(f'{gene_name}, pop {pop}\ngen {gen}', fontsize=10)
        ax.set_xlabel('SNP index')
        ax.set_ylabel('SNP index')
        plt.colorbar(im, ax=ax, label='r²', fraction=0.046, pad=0.04)

    plt.suptitle(f'LD (r²) buildup over generations — {gene_name}, pop {pop}',
                 fontsize=12, y=1.02)
    plt.tight_layout()
    plt.show()


# ============================================================
# 6.  GRID PLOT  (unchanged)
# ============================================================

def state_to_grid_result(state: dict) -> dict:
    result: dict = {}
    for pop, genes in state.items():
        result[pop] = {g: d['rgb'] for g, d in genes.items()}
    return result


def plot_gene_grid(result: dict, k: int, genes_per_deme: int = 4):
    n_cols = int(np.sqrt(genes_per_deme))
    cell_size = 1.0 / n_cols
    fig, ax = plt.subplots(figsize=(k * 1.5, k * 1.5))
    fig.subplots_adjust(right=0.78)

    for m in range(k * k):
        pop = m + 1
        r, c = m // k, m % k
        cx, cy = c, k - 1 - r
        gene_colors = []
        for gene_name in [f'gene_{i}' for i in range(1, genes_per_deme + 1)]:
            if pop in result and gene_name in result[pop]:
                gene_colors.append(result[pop][gene_name].mean(axis=0) / 255.0)
            else:
                gene_colors.append([0.5, 0.5, 0.5])

        for idx in range(genes_per_deme):
            sr, sc = idx // n_cols, idx % n_cols
            x = cx + sc * cell_size
            y = cy + (n_cols - 1 - sr) * cell_size
            ax.add_patch(patches.Rectangle(
                (x, y), cell_size, cell_size,
                color=gene_colors[idx], ec='white', lw=0.3))

        ax.add_patch(patches.Rectangle(
            (cx, cy), 1, 1, fill=False, ec='black', lw=1.5))

    ax.set_xlim(0, k); ax.set_ylim(0, k)
    ax.set_aspect('equal'); ax.axis('off')
    ax.set_title(
        f'Generation grid — {genes_per_deme} genes per deme ({n_cols}×{n_cols})',
        fontsize=12)

    inset = fig.add_axes([0.80, 0.80, 0.10, 0.10])
    colors_key = ['#4444cc', '#cc8844', '#44aa44', '#cc4444']
    for idx in range(genes_per_deme):
        sr, sc = idx // n_cols, idx % n_cols
        inset.add_patch(patches.Rectangle(
            (sc, n_cols - 1 - sr), 1, 1,
            color=colors_key[idx], ec='white', lw=1))
        inset.text(sc + 0.5, n_cols - 1 - sr + 0.5, f'gene_{idx+1}',
                   ha='center', va='center', fontsize=7,
                   color='white', fontweight='bold')
    inset.set_xlim(0, n_cols); inset.set_ylim(0, n_cols)
    inset.set_aspect('equal'); inset.axis('off')
    inset.set_title('Position key', fontsize=8, pad=4)
    plt.show()


# ============================================================
# 7.  FULL SIMULATION LOOP
# ============================================================

def _make_grid_fig(result: dict, k: int, gen: int,
                   genes_per_deme: int = 4) -> plt.Figure:
    """Build a grid figure without showing it. Returns the Figure."""
    n_cols = int(np.sqrt(genes_per_deme))
    cell_size = 1.0 / n_cols
    fig, ax = plt.subplots(figsize=(k * 1.5, k * 1.5))
    fig.subplots_adjust(right=0.78)
    for m in range(k * k):
        pop = m + 1
        r, c = m // k, m % k
        cx, cy = c, k - 1 - r
        gene_colors = []
        for gene_name in [f'gene_{i}' for i in range(1, genes_per_deme + 1)]:
            if pop in result and gene_name in result[pop]:
                gene_colors.append(result[pop][gene_name].mean(axis=0) / 255.0)
            else:
                gene_colors.append([0.5, 0.5, 0.5])
        for idx in range(genes_per_deme):
            sr, sc = idx // n_cols, idx % n_cols
            x = cx + sc * cell_size
            y = cy + (n_cols - 1 - sr) * cell_size
            ax.add_patch(patches.Rectangle(
                (x, y), cell_size, cell_size,
                color=gene_colors[idx], ec='white', lw=0.3))
        ax.add_patch(patches.Rectangle(
            (cx, cy), 1, 1, fill=False, ec='black', lw=1.5))
    ax.set_xlim(0, k); ax.set_ylim(0, k)
    ax.set_aspect('equal'); ax.axis('off')
    ax.set_title(f'Gen {gen} — {genes_per_deme} genes per deme ({n_cols}×{n_cols})',
                 fontsize=12)
    inset = fig.add_axes([0.80, 0.80, 0.10, 0.10])
    colors_key = ['#4444cc', '#cc8844', '#44aa44', '#cc4444']
    for idx in range(genes_per_deme):
        sr, sc = idx // n_cols, idx % n_cols
        inset.add_patch(patches.Rectangle(
            (sc, n_cols - 1 - sr), 1, 1,
            color=colors_key[idx], ec='white', lw=1))
        inset.text(sc + 0.5, n_cols - 1 - sr + 0.5, f'gene_{idx+1}',
                   ha='center', va='center', fontsize=7,
                   color='white', fontweight='bold')
    inset.set_xlim(0, n_cols); inset.set_ylim(0, n_cols)
    inset.set_aspect('equal'); inset.axis('off')
    inset.set_title('Position key', fontsize=8, pad=4)
    plt.close(fig)
    return fig


def _make_ld_fig(haplotypes: dict, gene_name: str, pop: int,
                 max_snps: int, gen: int) -> plt.Figure:
    """Build an LD heatmap figure without showing it. Returns the Figure."""
    r2 = compute_ld(haplotypes, gene_name, pop, max_snps)
    fig, ax = plt.subplots(figsize=(6, 5))
    im = ax.imshow(r2, cmap='Reds', vmin=0, vmax=1, aspect='auto')
    plt.colorbar(im, ax=ax, label='r²')
    ax.set_title(f'LD (r²) — {gene_name}, pop {pop}, gen {gen}', fontsize=11)
    ax.set_xlabel('SNP index'); ax.set_ylabel('SNP index')
    plt.tight_layout()
    plt.close(fig)
    return fig


def run_simulation(
    gene_dfs: dict,
    fitness_targets: dict,
    n_generations: int = 20,
    recombination_rate: float = 1.0,
    gene_order: list = None,
    neighbors: dict = None,
    migration_rate: float = 0.05,
    k: int = 10,
    plot_every: int = 5,
    plot_ld_gene: Optional[str] = None,
    plot_ld_pop: Optional[int] = None,
    max_ld_snps: int = 100,
):
    """
    Run a full forward-in-time simulation.

    No plots are shown during the run. Figures are collected and returned.
    Only the fitness trajectory is plotted live at the end.

    Parameters
    ----------
    gene_dfs          : your existing gene_dfs dict
    fitness_targets   : {(pop, gene_name): (R, G, B)}
    n_generations     : how many generations to simulate
    recombination_rate: probability of crossover per gamete (default 1.0)
    gene_order        : gene names in chromosome order (default gene_1→4)
    neighbors         : dict from create_grid(). Pass None for no migration.
    migration_rate    : fraction of each deme replaced by migrants (default 0.05)
    k                 : grid size (k×k populations)
    plot_every        : collect grid + LD figure every N generations (0 = never)
    plot_ld_gene      : gene name to collect LD figures for, or None
    plot_ld_pop       : population to collect LD figures for, or None
    max_ld_snps       : max SNPs for LD heatmap subsampling (default 100)

    Returns
    -------
    haplotypes  : final generation haplotypes
    state       : final GenerationState
    gen0_pcas   : stored PCA projectors
    history     : list of dicts with per-generation summary stats
    figs_grid   : list of (gen, Figure) — grid snapshots, not yet shown
    figs_ld     : list of (gen, Figure) — LD snapshots, not yet shown
    """
    print("Phasing diploid data into haplotypes...")
    haplotypes = phase_to_haplotypes(gene_dfs)

    print("Fitting gen-0 PCA projectors...")
    diploid_dfs = haplotypes_to_diploid(haplotypes)
    gen0_pcas, state = fit_gen0(diploid_dfs, fitness_targets)

    figs_grid: list = []
    figs_ld:   list = []

    figs_grid.append((0, _make_grid_fig(state_to_grid_result(state), k, gen=0)))
    if plot_ld_gene and plot_ld_pop:
        figs_ld.append((0, _make_ld_fig(haplotypes, plot_ld_gene,
                                         plot_ld_pop, max_ld_snps, gen=0)))
    history = []

    for gen in range(1, n_generations + 1):
        haplotypes = reproduce(
            haplotypes, state,
            recombination_rate=recombination_rate,
            gene_order=gene_order,
            neighbors=neighbors,
            migration_rate=migration_rate,
        )
        diploid_dfs = haplotypes_to_diploid(haplotypes)
        state = compute_state(diploid_dfs, gen0_pcas, fitness_targets)

        gen_stats = {'generation': gen}
        for (pop, gene_name), target in fitness_targets.items():
            f = state[pop][gene_name]['fitness']
            if f is not None:
                gen_stats[f'mean_fitness_pop{pop}_{gene_name}'] = f.mean()
        history.append(gen_stats)

        print(f"Gen {gen:3d} | " + " | ".join(
            f"{key}: {val:.3f}"
            for key, val in gen_stats.items() if key != 'generation'
        ))

        if plot_every > 0 and (gen % plot_every == 0 or gen == n_generations):
            figs_grid.append((gen, _make_grid_fig(
                state_to_grid_result(state), k, gen=gen)))
            if plot_ld_gene and plot_ld_pop:
                figs_ld.append((gen, _make_ld_fig(
                    haplotypes, plot_ld_gene, plot_ld_pop, max_ld_snps, gen=gen)))

    # ── fitness trajectory — only live plot shown ──
    hist_df = pd.DataFrame(history).set_index('generation')
    fig_fit, ax_fit = plt.subplots(figsize=(10, 4))
    for col in hist_df.columns:
        ax_fit.plot(hist_df.index, hist_df[col], label=col)
    ax_fit.set_xlabel('Generation')
    ax_fit.set_ylabel('Mean fitness')
    ax_fit.set_title('Fitness trajectory')
    ax_fit.legend(fontsize=8, bbox_to_anchor=(1.01, 1), loc='upper left')
    ax_fit.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

    return haplotypes, state, gen0_pcas, history, figs_grid, figs_ld


# ============================================================
# USAGE EXAMPLE
# ============================================================
#
# _, neighbors = create_grid(k=10)   # from your existing code
#
# fitness_targets = {
#     (1,  'gene_1'): (0,   255, 0),    # pop 1  gene_1 → pure green
#     (5,  'gene_2'): (255, 0,   0),    # pop 5  gene_2 → pure red
#     (10, 'gene_3'): (0,   0, 255),    # pop 10 gene_3 → pure blue
# }
#
# haplotypes, state, gen0_pcas, history = run_simulation(
#     gene_dfs           = gene_dfs,
#     fitness_targets    = fitness_targets,
#     n_generations      = 30,
#     recombination_rate = 1.0,
#     neighbors          = neighbors,   # pass None to disable migration
#     migration_rate     = 0.05,        # 5% of each deme are migrants
#     k                  = 10,
#     plot_every         = 5,
#     plot_ld_gene       = 'gene_1',
#     plot_ld_pop        = 1,
# )
#
# # Plot fitness trajectory
# pd.DataFrame(history).set_index('generation').plot(figsize=(10, 4))
# plt.title('Mean fitness per tracked (pop, gene) over generations')
# plt.ylabel('Mean fitness')
# plt.show()

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt

_, neighbors = create_grid(k=10)   # from your existing code

n_gens = 500
haplotypes, state, gen0_pcas, history, figs_grid, figs_ld = run_simulation(
    gene_dfs           = gene_dfs,
    fitness_targets    = fitness_targets,
    n_generations      = n_gens,
    recombination_rate = 1.0,
    neighbors          = neighbors,
    migration_rate     = 0.05,
    k                  = 10,
    plot_every         = 5,
    plot_ld_gene       = 'gene_1',
    plot_ld_pop        = 1,
)


In [ ]:
from IPython.display import display

for gen, fig in figs_grid:
    display(fig)

for gen, fig in figs_ld:
    display(fig)

In [ ]:
final_gene_dfs = haplotypes_to_diploid(haplotypes)
final_genotype = gene_dfs_to_genotype_df(final_gene_dfs)

In [ ]:
snap_keys = sorted(snapshots.keys())
print(snap_keys)  # e.g. [0, 33, 66]

compare_ld_generations(
    haplotypes_list = [snapshots[k] for k in snap_keys],
    generations     = snap_keys,
    pop             = 1,
    gene_name       = 'gene_1',
)

r2_matrix, decay_df = plot_genome_wide_ld(
    snapshots[snap_keys[-1]], pop=1, generation=snap_keys[-1]
)

In [ ]:
snap_keys = [0, int(n_gens/3), int(2*n_gens/3)]

compare_ld_generations(
    haplotypes_list = [snapshots[k] for k in snap_keys],
    generations     = snap_keys,
    pop             = 1,
    gene_name       = 'gene_1',
)

r2_matrix, decay_df = plot_genome_wide_ld(
    snapshots[int(2*n_gens/3)], pop=1, generation=int(2*n_gens/3)
)